# Глава 9: Слой эмбеддингов

> "Слово познаётся по компании, которую оно составляет." — **Джон Руперт Ферс**, лингвист

---

## Что вы узнаете

- Почему одних идентификаторов токенов недостаточно и какую проблему на самом деле решают эмбеддинги
- Как создать эмбеддинги токенов с нуля, используя таблицы поиска
- Почему важна позиционная информация и как её добавить
- Как объединить эмбеддинги токенов и позиций для полного представления входных данных
- Как исследовать эмбеддинги реальных моделей и обнаруживать семантические отношения
- Практические аспекты инициализации и реализации

---

## Настройка

Сначала установим необходимые пакеты:

In [ ]:
# Установка необходимых пакетов
!pip install -q torch transformers

In [ ]:
# ===== ИМПОРТЫ =====
import torch                     # PyTorch: операции с тензорами и нейронные сети
import torch.nn as nn            # Модули нейронных сетей (слои и т.д.)
import torch.nn.functional as F  # Математические функции (cosine_similarity и т.д.)
from transformers import AutoModel, AutoTokenizer  # Предобученные модели

# Краткое напоминание о тензорах:
# - torch.tensor([1,2,3]) создаёт одномерный тензор (как список)
# - torch.randn(3, 4) создаёт тензор 3×4 со случайными числами
# - tensor[0] обращается к первому измерению
# - tensor.shape показывает размерности

## 1. Почему нельзя просто использовать идентификаторы токенов?

Идентификаторы токенов (token IDs) — это произвольные целые числа без семантического значения. Давайте посмотрим на проблему:

In [ ]:
# Идентификаторы токенов — это просто целые числа без связей
# (настоящие ID GPT-2; начальные пробелы важны: " cat" против "cat")
token_ids = {
    " cat": 3797,
    " kitten": 42143,
    " dog": 3290,
    " car": 1097
}

print("Идентификаторы токенов:")
for word, id in token_ids.items():
    print(f"  '{word}' → {id}")

# Проблема: "cat" численно ближе к "dog", чем к "kitten"!
print(f"\nРасстояние от 'cat' до 'dog': {abs(3797 - 3290)}")
print(f"Расстояние от 'cat' до 'kitten': {abs(3797 - 42143)}")
print("\nНо семантически 'cat' и 'kitten' более похожи!")

### Решение: эмбеддинги

Преобразуем каждый идентификатор токена в плотный вектор, который захватывает смысл.

**Что такое `nn.Embedding`?**
- Таблица поиска с `vocab_size` строками и `embed_dim` столбцами
- Каждая строка — это вектор, представляющий один токен
- Вход: идентификатор токена (целое число) → Выход: эта строка (вектор)
- Представьте себе словарь: `{0: [0.1, 0.2, ...], 1: [0.5, -0.1, ...], ...}`

In [ ]:
# ===== Проблема: идентификаторы токенов =====
token_ids = torch.tensor([3797, 42143, 3290])  # " cat", " kitten", " dog"
print(f"Форма идентификаторов токенов: {token_ids.shape}")
print(f"Просто целые числа: {token_ids}")

# ===== Решение: эмбеддинги =====
vocab_size = 50257  # Словарь GPT-2
embed_dim = 768     # Размерность эмбеддинга GPT-2

# Создаём слой эмбеддингов (это таблица поиска!)
embedding = nn.Embedding(vocab_size, embed_dim)

# Ищем векторы для наших идентификаторов токенов
token_vectors = embedding(token_ids)
print(f"\nФорма эмбеддингов токенов: {token_vectors.shape}")
print(f"Вектор первого токена (первые 10 измерений): {token_vectors[0, :10]}")

# Теперь каждый токен — это 768-мерный вектор, который может захватывать смысл!

## 2. Эмбеддинги токенов: таблица поиска

Давайте построим эмбеддинги токенов с нуля, чтобы понять механизм.

### Шаг 1: создание матрицы эмбеддингов

In [ ]:
vocab_size = 50257  # Размер словаря GPT-2
embed_dim = 768     # Размерность эмбеддинга

# Создаём матрицу эмбеддингов: одна строка на токен
embedding_matrix = torch.randn(vocab_size, embed_dim)

print(f"Форма матрицы эмбеддингов: {embedding_matrix.shape}")
print(f"\nЭмбеддинг токена 3797 (первые 10 измерений): {embedding_matrix[3797, :10]}")

### Шаг 2: поиск нескольких токенов

In [ ]:
token_ids = torch.tensor([464, 3797, 3332])  # "The cat sat"

# Ручной поиск (то, что делают слои эмбеддингов внутри)
embeddings = embedding_matrix[token_ids]

print(f"Форма входа: {token_ids.shape}")       # torch.Size([3])
print(f"Форма выхода: {embeddings.shape}")     # torch.Size([3, 768])

# Каждый идентификатор токена → его 768-мерный вектор
print(f"\nЭмбеддинг токена 464 (первые 5 измерений): {embeddings[0, :5]}")
print(f"Эмбеддинг токена 3797 (первые 5 измерений): {embeddings[1, :5]}")
print(f"Эмбеддинг токена 3332 (первые 5 измерений): {embeddings[2, :5]}")

### Шаг 3: обработка батчей

In [ ]:
# Батч из 2 последовательностей, каждая с 4 токенами
token_ids_batch = torch.tensor([
    [464, 3797, 3332, 319],    # Последовательность 1: "The cat sat on"
    [40, 481, 1037, 345]       # Последовательность 2: "I will help you"
])

print(f"Форма батча: {token_ids_batch.shape}")  # torch.Size([2, 4])

# Ищем эмбеддинги для всего батча
embeddings_batch = embedding_matrix[token_ids_batch]

print(f"Форма эмбеддингов: {embeddings_batch.shape}")  # torch.Size([2, 4, 768])
print("\nПреобразование формы: (batch, seq) → (batch, seq, embed_dim)")

### Шаг 4: использование nn.Embedding из PyTorch

In [ ]:
class TokenEmbedding(nn.Module):
    """
    Слой эмбеддингов токенов: преобразует идентификаторы токенов в плотные векторы.
    
    Это то, что делает слой 'wte' (word token embeddings) в GPT-2.
    """
    def __init__(self, vocab_size, embed_dim):
        super().__init__()
        # Создаём матрицу эмбеддингов как обучаемый параметр
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        
        # Инициализируем малыми случайными значениями (стиль GPT-2)
        nn.init.normal_(self.embedding.weight, mean=0.0, std=0.02)
    
    def forward(self, token_ids):
        """
        Аргументы:
            token_ids: тензор идентификаторов токенов (batch, seq)
        
        Возвращает:
            embeddings: тензор векторов токенов (batch, seq, embed_dim)
        """
        return self.embedding(token_ids)

# Создаём слой эмбеддингов токенов
token_embed = TokenEmbedding(vocab_size=50257, embed_dim=768)

# Получаем эмбеддинги для батча
token_ids = torch.tensor([[464, 3797, 3332, 319]])  # Форма: (1, 4)
embeddings = token_embed(token_ids)

print(f"Форма входа: {token_ids.shape}")        # torch.Size([1, 4])
print(f"Форма выхода: {embeddings.shape}")      # torch.Size([1, 4, 768])
print(f"\nЭмбеддинг первого токена (первые 5 измерений): {embeddings[0, 0, :5]}")

## 3. Позиционные эмбеддинги: обучение позиции

Эмбеддинги токенов не содержат информации о позиции. Давайте посмотрим на проблему:

In [ ]:
# Две последовательности с одинаковыми токенами, но разным порядком
token_ids_1 = torch.tensor([[464, 3797, 3332]])  # "The cat sat"
token_ids_2 = torch.tensor([[3332, 3797, 464]])  # "sat cat The"

# Получаем эмбеддинги токенов
token_embed = TokenEmbedding(vocab_size=50257, embed_dim=768)
embeddings_1 = token_embed(token_ids_1)
embeddings_2 = token_embed(token_ids_2)

print(f"Форма эмбеддингов 1: {embeddings_1.shape}")
print(f"Форма эмбеддингов 2: {embeddings_2.shape}")

# Они разные...
print(f"\nЭмбеддинги идентичны? {torch.equal(embeddings_1, embeddings_2)}")

# Но если отсортировать оба, они содержат одинаковые векторы!
# Это проблема: без информации о позиции
# "The cat sat" и "sat cat The" выглядят одинаково для слоёв внимания

### Реализация обучаемых позиционных эмбеддингов

In [ ]:
class PositionalEmbedding(nn.Module):
    """
    Обучаемые позиционные эмбеддинги: один обучаемый вектор на позицию.
    
    GPT-2 использует этот подход (называется 'wpe' - word position embeddings).
    """
    def __init__(self, max_seq_len, embed_dim):
        """
        Аргументы:
            max_seq_len: Максимальная длина последовательности (например, 1024 для GPT-2)
            embed_dim: Размерность эмбеддинга (должна совпадать с эмбеддингами токенов)
        """
        super().__init__()
        # Создаём матрицу позиционных эмбеддингов: (max_seq_len, embed_dim)
        self.pos_embed = nn.Embedding(max_seq_len, embed_dim)
        
        # Инициализируем малыми случайными значениями (стиль GPT-2)
        nn.init.normal_(self.pos_embed.weight, mean=0.0, std=0.02)
        
        self.max_seq_len = max_seq_len
    
    def forward(self, token_ids):
        """
        Аргументы:
            token_ids: тензор идентификаторов токенов (batch, seq)
        
        Возвращает:
            pos_embeddings: тензор векторов позиций (batch, seq, embed_dim)
        """
        batch_size, seq_len = token_ids.shape
        
        # Проверяем длину последовательности
        if seq_len > self.max_seq_len:
            raise ValueError(
                f"Длина последовательности {seq_len} превышает max_seq_len {self.max_seq_len}"
            )
        
        # Создаём индексы позиций: [0, 1, 2, ..., seq_len-1]
        position_ids = torch.arange(
            seq_len,
            device=token_ids.device  # Сопоставляем устройство (CPU/GPU) входа
        )
        
        # Расширяем для батча: (seq_len,) → (batch, seq_len)
        position_ids = position_ids.unsqueeze(0).expand(batch_size, seq_len)
        
        # Ищем позиционные эмбеддинги
        pos_embeddings = self.pos_embed(position_ids)
        
        return pos_embeddings

# Создаём слой позиционных эмбеддингов
pos_embed = PositionalEmbedding(max_seq_len=1024, embed_dim=768)

# Пример: последовательность длиной 4
token_ids = torch.tensor([[464, 3797, 3332, 319]])  # Форма: (1, 4)
pos_embeddings = pos_embed(token_ids)

print(f"Форма входа: {token_ids.shape}")           # torch.Size([1, 4])
print(f"Форма позиционных эмбеддингов: {pos_embeddings.shape}")  # torch.Size([1, 4, 768])

# Каждая позиция получает свой обученный вектор
print(f"\nЭмбеддинг позиции 0 (первые 5 измерений): {pos_embeddings[0, 0, :5]}")
print(f"Эмбеддинг позиции 1 (первые 5 измерений): {pos_embeddings[0, 1, :5]}")
print(f"Эмбеддинг позиции 2 (первые 5 измерений): {pos_embeddings[0, 2, :5]}")
print(f"Эмбеддинг позиции 3 (первые 5 измерений): {pos_embeddings[0, 3, :5]}")

## 4. Объединение эмбеддингов токенов и позиций

### Построение полного класса GPT2Embeddings

In [ ]:
class GPT2Embeddings(nn.Module):
    """
    Полный слой эмбеддингов GPT-2: эмбеддинги токенов + позиций.
    
    Это соответствует слоям 'wte' + 'wpe' в GPT-2.
    """
    def __init__(self, vocab_size, max_seq_len, embed_dim):
        """
        Аргументы:
            vocab_size: Размер словаря (50257 для GPT-2)
            max_seq_len: Максимальная длина последовательности (1024 для GPT-2)
            embed_dim: Размерность эмбеддинга (768 для GPT-2 Small)
        """
        super().__init__()
        
        # Эмбеддинги токенов: vocab_size × embed_dim
        self.token_embed = nn.Embedding(vocab_size, embed_dim)
        
        # Позиционные эмбеддинги: max_seq_len × embed_dim
        self.pos_embed = nn.Embedding(max_seq_len, embed_dim)
        
        # Инициализируем оба стандартной инициализацией GPT-2
        nn.init.normal_(self.token_embed.weight, mean=0.0, std=0.02)
        nn.init.normal_(self.pos_embed.weight, mean=0.0, std=0.02)
        
        self.max_seq_len = max_seq_len
    
    def forward(self, token_ids):
        """
        Аргументы:
            token_ids: тензор идентификаторов токенов (batch, seq)
        
        Возвращает:
            embeddings: тензор объединённых эмбеддингов (batch, seq, embed_dim)
        """
        batch_size, seq_len = token_ids.shape
        
        # Проверяем длину последовательности
        if seq_len > self.max_seq_len:
            raise ValueError(
                f"Длина последовательности {seq_len} превышает max_seq_len {self.max_seq_len}"
            )
        
        # ===== Эмбеддинги токенов =====
        token_embeddings = self.token_embed(token_ids)
        
        # ===== Позиционные эмбеддинги =====
        position_ids = torch.arange(seq_len, device=token_ids.device)
        position_ids = position_ids.unsqueeze(0).expand(batch_size, seq_len)
        position_embeddings = self.pos_embed(position_ids)
        
        # ===== Объединяем через сложение =====
        embeddings = token_embeddings + position_embeddings
        
        return embeddings

# Создаём слой эмбеддингов GPT-2 Small
gpt2_embed = GPT2Embeddings(
    vocab_size=50257,
    max_seq_len=1024,
    embed_dim=768
)

# Пример: получаем эмбеддинги для батча последовательностей (ОБЕ должны иметь одинаковую длину!)
# У GPT-2 нет специального pad-токена: ID 0 — это '!', а не заполнение.
# Общепринятый подход (глава 12) — дополнять токеном конца текста, ID 50256.
token_ids = torch.tensor([
    [464, 3797, 3332, 319, 50256, 50256],  # "The cat sat on" + заполнение
    [40, 481, 1037, 345, 50256, 50256]     # "I will help you" + заполнение
])

embeddings = gpt2_embed(token_ids)

print(f"Форма входа: {token_ids.shape}")         # torch.Size([2, 6])
print(f"Форма выхода: {embeddings.shape}")       # torch.Size([2, 6, 768])
print(f"Каждый токен теперь имеет: {embeddings.shape[-1]} измерений")

print(f"\nПервая последовательность, первый токен (первые 10 измерений):")
print(embeddings[0, 0, :10])

### Связь с главой 8: полный конвейер

In [ ]:
from transformers import AutoTokenizer

# ===== Шаг 1: токенизация (глава 8) =====
tokenizer = AutoTokenizer.from_pretrained("gpt2")
text = "The cat sat on the mat"
token_ids = tokenizer.encode(text, return_tensors="pt")

print("Шаг 1: Токенизация")
print(f"Текст: {text}")
print(f"Идентификаторы токенов: {token_ids}")
print(f"Форма: {token_ids.shape}\n")

# ===== Шаг 2: эмбеддинги (глава 9) =====
gpt2_embed = GPT2Embeddings(vocab_size=50257, max_seq_len=1024, embed_dim=768)
embeddings = gpt2_embed(token_ids)

print("Шаг 2: Эмбеддинги")
print(f"Форма эмбеддингов: {embeddings.shape}")
print(f"Эмбеддинг первого токена (первые 10 измерений): {embeddings[0, 0, :10]}\n")

print("Шаг 3: Далее — слои внимания (глава 10)")
print(f"Эти эмбеддинги формы {embeddings.shape} будут поступать в самовнимание,")
print("где токены будут учиться из контекста друг друга!")

## 5. Исследование эмбеддингов GPT-2

Загрузим реальную предобученную модель GPT-2 и исследуем, чему она научилась:

In [ ]:
# Загружаем GPT-2 Small
model = AutoModel.from_pretrained("gpt2")
tokenizer = AutoTokenizer.from_pretrained("gpt2")

# Получаем доступ к эмбеддингам токенов
token_embeddings = model.wte.weight  # Word Token Embeddings
print(f"Форма эмбеддингов токенов: {token_embeddings.shape}")
# torch.Size([50257, 768]) — один 768-мерный вектор на токен

# Получаем доступ к позиционным эмбеддингам
position_embeddings = model.wpe.weight  # Word Position Embeddings
print(f"Форма позиционных эмбеддингов: {position_embeddings.shape}")
# torch.Size([1024, 768]) — один 768-мерный вектор на позицию

### Поиск похожих слов

**Что такое косинусное сходство (cosine similarity)?**
Измеряет, насколько похожи два вектора, на основе угла между ними:
- **1.0** = одинаковое направление (очень похожи)
- **0.0** = перпендикулярны (не связаны)
- **-1.0** = противоположное направление (противоположный смысл)

Представьте это так: "эти две стрелки указывают в одном направлении?"

In [ ]:
def find_similar_tokens(word, embeddings, tokenizer, top_k=5):
    """Находит токены с эмбеддингами, наиболее похожими на заданное слово."""
    # Получаем идентификатор токена для слова
    token_id = tokenizer.encode(word, add_special_tokens=False)[0]
    target_vec = embeddings[token_id]
    
    # Вычисляем косинусное сходство со всеми токенами
    similarities = F.cosine_similarity(
        target_vec.unsqueeze(0),  # (1, 768)
        embeddings,               # (50257, 768)
        dim=1
    )
    
    # Получаем top-k наиболее похожих (исключая само слово)
    top_indices = similarities.argsort(descending=True)[1:top_k+1]
    
    print(f"\nСлова, наиболее похожие на '{word}':")
    for idx in top_indices:
        token = tokenizer.decode([idx])
        score = similarities[idx].item()
        print(f"  {score:.3f} — '{token}'")

# Исследуем семантические отношения
find_similar_tokens("king", token_embeddings, tokenizer, top_k=8)
find_similar_tokens("computer", token_embeddings, tokenizer, top_k=8)
find_similar_tokens("happy", token_embeddings, tokenizer, top_k=8)

### Попробуйте свои собственные исследования!

In [ ]:
# Попробуйте разные слова:
words_to_explore = ["Python", "doctor", "fast", "beautiful"]

for word in words_to_explore:
    find_similar_tokens(word, token_embeddings, tokenizer, top_k=5)

### Бонус: словесные аналогии

Можем ли мы найти векторы, удовлетворяющие уравнению "king - man + woman ≈ queen"?

In [ ]:
def word_analogy(word1, word2, word3, embeddings, tokenizer, top_k=5):
    """
    Находит: word1 - word2 + word3 ≈ ?
    Пример: king - man + woman ≈ queen
    """
    # Получаем идентификаторы токенов
    id1 = tokenizer.encode(word1, add_special_tokens=False)[0]
    id2 = tokenizer.encode(word2, add_special_tokens=False)[0]
    id3 = tokenizer.encode(word3, add_special_tokens=False)[0]
    
    # Вычисляем целевой вектор: word1 - word2 + word3
    target_vec = embeddings[id1] - embeddings[id2] + embeddings[id3]
    
    # Находим наиболее похожие токены
    similarities = F.cosine_similarity(
        target_vec.unsqueeze(0),
        embeddings,
        dim=1
    )
    
    # Исключаем входные слова из результатов
    similarities[id1] = -1
    similarities[id2] = -1
    similarities[id3] = -1
    
    top_indices = similarities.argsort(descending=True)[:top_k]
    
    print(f"\n{word1} - {word2} + {word3} ≈ ?")
    for idx in top_indices:
        token = tokenizer.decode([idx])
        score = similarities[idx].item()
        print(f"  {score:.3f} — '{token}'")

# Классический пример: king - man + woman ≈ queen
word_analogy("king", "man", "woman", token_embeddings, tokenizer, top_k=5)

# Попробуйте другие!
word_analogy("Paris", "France", "Germany", token_embeddings, tokenizer, top_k=5)

## 6. Практические упражнения

### Упражнение 1: ручной поиск эмбеддингов

```python
# Создайте небольшой словарь (10 токенов) и размерность эмбеддинга 8
# Вручную создайте матрицу эмбеддингов и найдите эмбеддинги для идентификаторов токенов [2, 5, 7]
# Выведите формы на каждом шаге

# Шаг 1: Создание матрицы эмбеддингов
vocab_size = 10
embed_dim = 8
embedding_matrix = torch.randn(???, ???)  # Заполните размеры
print(f"Форма матрицы эмбеддингов: {embedding_matrix.shape}")

# Шаг 2: Создание идентификаторов токенов для поиска
token_ids = torch.tensor([2, 5, 7])
print(f"Идентификаторы токенов: {token_ids}")

# Шаг 3: Поиск эмбеддингов (подсказка: используйте индексацию как embedding_matrix[...])
embeddings = ???
print(f"Форма эмбеддингов: {embeddings.shape}")
```

### Упражнение 2: создание TokenEmbedding с нуля

In [ ]:
# Реализуйте класс TokenEmbedding, не глядя на пример
# Включите правильную инициализацию и проверку формы

# ВАШ КОД ЗДЕСЬ

### Упражнение 3: позиционное кодирование

In [ ]:
# Создайте последовательность идентификаторов токенов: [10, 20, 30, 40, 50]
# Сгенерируйте индексы позиций и найдите их в слое позиционных эмбеддингов
# Проверьте, что позиция 0 всегда получает один и тот же вектор независимо от токена

# ВАШ КОД ЗДЕСЬ

### Упражнение 4: полный GPT2Embeddings

In [ ]:
# Реализуйте полный класс GPT2Embeddings
# Протестируйте его с:
# - Батчем из 2 последовательностей
# - Разными длинами последовательностей (одна длиной 5, другая длиной 8 с заполнением)
# - Проверьте, что форма выхода (2, 8, embed_dim)

# ВАШ КОД ЗДЕСЬ

### Упражнение 5: исследование сходств в GPT-2

In [ ]:
# Загрузите GPT-2 и найдите токены, похожие на:
# - "Python" (должен найти слова, связанные с программированием)
# - "doctor" (должен найти медицинские/профессиональные слова)
# - "fast" (должен найти слова, связанные со скоростью)
#
# Группирует ли модель связанные концепции вместе?

# ВАШ КОД ЗДЕСЬ

### Упражнение 6: обработка устройств

In [ ]:
# Создайте экземпляр GPT2Embeddings
# Переместите его на GPU (если доступен)
# Получите эмбеддинги для идентификаторов токенов, которые начинаются на CPU
# Что произойдёт? Исправьте это, сначала переместив идентификаторы токенов на GPU

# ВАШ КОД ЗДЕСЬ

### Упражнение 7: ограничения длины последовательности

In [ ]:
# Создайте GPT2Embeddings с max_seq_len=10
# Попробуйте получить эмбеддинги для последовательности длиной 15
# Обработайте ошибку корректно, обрезав последовательность перед получением эмбеддингов

# ВАШ КОД ЗДЕСЬ

### Упражнение 8: интеграция с главой 8

In [ ]:
# Возьмите выходной файл JSONL из главы 8 (ваш токенизированный набор данных)
# Загрузите одну запись, извлеките идентификаторы токенов, преобразуйте в тензор PyTorch
# Получите эмбеддинги, используя GPT2Embeddings
# Выведите форму на каждом шаге

# ВАШ КОД ЗДЕСЬ

## Резюме главы

**Что мы построили:**

1. **Эмбеддинги токенов:** Таблица поиска, преобразующая идентификаторы токенов в семантические векторы
2. **Позиционные эмбеддинги:** Обученные векторы, кодирующие позицию в последовательности
3. **Полный GPT2Embeddings:** Объединяет токены и позиции через сложение

**Что мы узнали:**

- Идентификаторы токенов — это произвольные индексы без семантического значения
- Эмбеддинги преобразуют идентификаторы в плотные векторы, которые захватывают отношения
- Информация о позиции критична ("cat chased dog" ≠ "dog chased cat")
- GPT-2 использует обучаемые позиционные эмбеддинги (обучаемые параметры)
- Косинусное сходство раскрывает изученные семантические отношения
- Эмбеддинги токенов GPT-2 в основном группируют варианты написания/регистра ("King", "KING"); более богатый смысл появляется после слоёв внимания

**Далее:** Глава 10 будет использовать эти эмбеддинги для самовнимания, где токены учатся из контекста друг друга!